# Train and reload ModPINN

Follow the [setup instructions](README.md), select the ModPINN kernel, and run cells in order. This three-step CPU example checks training and reload. [Standalone script](../quickstart.md).

In [ ]:
from pathlib import Path
from uuid import uuid4
import torch
from modpinn.training import catalogue, train, evaluate, load_pretrained

torch.set_num_threads(1)
print(len(catalogue()), "available training configurations")

## Train

Smoke mode uses three SOAP steps on an 8-by-8 grid. Each execution creates a fresh directory under `datasets/runs/`, relative to the source root or current directory for an installed-only example.

In [ ]:
workspace = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src/modpinn").is_dir() and (p / "datasets/experiments.json").is_file()),
    Path.cwd(),
)
output = workspace / "datasets/runs" / f"notebook-{uuid4().hex[:8]}"
result = train("best/amplitude-0.1", output, smoke=True)
assert result["epochs_completed"] == result["adaptive_updates"] == 3
assert result["parameter_change_l2"] > 0
print("Output:", output)
print(result)

## Reload and evaluate

The checkpoint stores its configuration. Evaluation uses bundled references; expect total error about 8.455 after three steps.

In [ ]:
metrics = evaluate(output / "model.pt")
print(metrics)
assert all(metrics[key] == result[key] for key in metrics if key.startswith("relative_l2_"))


## Predict with a bundled checkpoint

Input columns: time, radius. Outputs: scalar field, lapse, compactness.

In [ ]:
model = load_pretrained("best/amplitude-0.1")
points = torch.tensor([[0.1, 0.2], [1.0, 0.8]], dtype=model.DTYPE, requires_grad=True)
phi, alpha, compactness = model(points)
assert all(torch.isfinite(field).all() for field in (phi, alpha, compactness))
phi.sum().backward()
print("fields:", torch.cat((phi, alpha, compactness), dim=1).detach())
print("scalar-field derivatives:", points.grad)


## Next

Continue with [predictions and derivatives](inference.ipynb), or see [training settings](../training.md). Software is MIT-licensed; numerical data retain their citation-based terms.